# Data Quality Audit — PocketCoder Pretraining Corpus

Measures the quality and uniqueness of `Ananda100/python-clean-codeparrot` (the 2.96B-token
pretraining corpus) on a streamed sample, and reports everything needed for the paper's
**Corpus Statistics** table (Table 1) in one block:

| # | Metric | Why it matters |
|---|---|---|
| 1 | Syntactic validity | did the regex cleaning work — % of docs that `ast.parse` |
| 2 | Natural-language content | comments + docstrings as % of characters (the 16.0% figure) |
| 3 | Uniqueness | exact + near-duplicate rates (near = comments/whitespace stripped) |
| 4 | Length distribution | mean / median / p95 / min / max chars per document |
| 5 | Docstring coverage | % of functions carrying a docstring — curation quality signal |
| 6 | Function complexity | avg control-flow branches per function — real logic vs trivial snippets |
| 7 | Worst duplicate clusters | spot-check of the largest repeated documents |

Also saves the results to `corpus_quality_report.json` so the numbers are a citable artifact.
No GPU needed; ~2–4 min for a 5,000-doc sample. The `SyntaxWarning: invalid escape sequence`
spam during the run is harmless — it's Python compiling regex-heavy source files from the
corpus, not an error.


In [1]:
!pip install -q -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.6 MB/s eta 0:00:00


In [2]:
# ======================= CONFIG =======================
DATASET_ID  = "Ananda100/python-clean-codeparrot"
SPLIT       = "train"
N_DOCS      = 5000          # sample size; raise to 20000 for tighter estimates
OUT_JSON    = "corpus_quality_report.json"
# ======================================================
print(f"Auditing {DATASET_ID} [{SPLIT}] on {N_DOCS} streamed documents")

Auditing Ananda100/python-clean-codeparrot [train] on 5000 streamed documents


## Definitions — all quality metrics

In [3]:
import ast, hashlib, re, statistics, warnings
from collections import Counter

warnings.filterwarnings("ignore", category=SyntaxWarning)   # silence corpus escape-sequence spam


def nl_fraction(source):
    """Characters in comments and docstrings vs. total."""
    total = len(source)
    comment_chars = 0
    for line in source.split("\n"):
        idx = line.find("#")
        if idx != -1:
            comment_chars += len(line[idx:])
    docstring_chars = 0
    try:
        tree = ast.parse(source)
        for node in ast.walk(tree):
            if isinstance(node, (ast.Module, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                doc = ast.get_docstring(node)
                if doc:
                    docstring_chars += len(doc)
    except SyntaxError:
        pass
    nl = comment_chars + docstring_chars
    return {"comment_chars": comment_chars, "docstring_chars": docstring_chars,
            "code_chars": max(total - nl, 0), "total_chars": total}


def is_valid_python(s):
    try:
        ast.parse(s)
        return True
    except SyntaxError:
        return False


def normalize_code(s):
    """Strip comments and blank lines for near-duplicate hashing."""
    lines = [re.sub(r"#.*$", "", l).strip() for l in s.split("\n")]
    return "\n".join(l for l in lines if l)


print("definitions ready")

definitions ready


## Load a streamed sample (no full download)

In [4]:
from datasets import load_dataset

ds = load_dataset(DATASET_ID, split=SPLIT, streaming=True)

sample = next(iter(ds))
print("Fields:", list(sample.keys()))
FIELD_NAME = "content" if "content" in sample else list(sample.keys())[0]
print("Using field:", FIELD_NAME)

sources = []
for i, ex in enumerate(ds):
    if i >= N_DOCS:
        break
    sources.append(ex[FIELD_NAME])
print(f"Loaded {len(sources)} documents")

README.md:   0%|          | 0.00/289 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Fields: ['content']
Using field: content
Loaded 5000 documents


## Run the full audit

In [5]:
import json

n = len(sources)
report = {"dataset": DATASET_ID, "split": SPLIT, "n_docs": n}

print(f"\n{'='*60}")
print(f"QUALITY REPORT: {DATASET_ID}  (n={n} documents)")
print(f"{'='*60}")

# [1] Syntactic validity
valid = sum(is_valid_python(s) for s in sources)
report["validity_pct"] = 100 * valid / n
print(f"\n[1] Syntactic validity:  {valid}/{n} = {report['validity_pct']:.2f}%")

# [2] Natural-language content (char-weighted)
tot = {"comment_chars": 0, "docstring_chars": 0, "code_chars": 0, "total_chars": 0}
for s in sources:
    r = nl_fraction(s)
    for k in tot:
        tot[k] += r[k]
report["comments_pct"]   = 100 * tot["comment_chars"] / tot["total_chars"]
report["docstrings_pct"] = 100 * tot["docstring_chars"] / tot["total_chars"]
report["nl_total_pct"]   = report["comments_pct"] + report["docstrings_pct"]
print(f"\n[2] Natural-language content:")
print(f"    Comments:   {report['comments_pct']:.2f}%")
print(f"    Docstrings: {report['docstrings_pct']:.2f}%")
print(f"    NL total:   {report['nl_total_pct']:.2f}%")

# [3] Uniqueness / duplication
exact = Counter(hashlib.md5(s.encode()).hexdigest() for s in sources)
near  = Counter(hashlib.md5(normalize_code(s).encode()).hexdigest() for s in sources)
e_d = sum(c - 1 for c in exact.values() if c > 1)
n_d = sum(c - 1 for c in near.values() if c > 1)
report["unique_exact_pct"] = 100 * len(exact) / n
report["unique_near_pct"]  = 100 * len(near) / n
report["exact_dup_pct"]    = 100 * e_d / n
report["near_dup_pct"]     = 100 * n_d / n
print(f"\n[3] Uniqueness:")
print(f"    Unique (exact):           {len(exact)}/{n} = {report['unique_exact_pct']:.2f}%")
print(f"    Unique (near, code-only): {len(near)}/{n} = {report['unique_near_pct']:.2f}%")
print(f"    Exact duplicate rate: {report['exact_dup_pct']:.2f}%")
print(f"    Near duplicate rate:  {report['near_dup_pct']:.2f}%")

# [4] Length distribution
lens = sorted(len(s) for s in sources)
report["len_mean"]   = statistics.mean(lens)
report["len_median"] = statistics.median(lens)
report["len_p95"]    = lens[int(0.95 * n)]
report["len_min"], report["len_max"] = lens[0], lens[-1]
print(f"\n[4] Document length (characters):")
print(f"    mean={report['len_mean']:.0f}  median={report['len_median']:.0f}  "
      f"p95={report['len_p95']}  min={report['len_min']}  max={report['len_max']}")

# [5] Docstring coverage
tf = documented = 0
for s in sources:
    try:
        tree = ast.parse(s)
    except SyntaxError:
        continue
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            tf += 1
            if ast.get_docstring(node):
                documented += 1
report["n_functions"] = tf
report["docstring_coverage_pct"] = 100 * documented / tf if tf else 0
print(f"\n[5] Docstring coverage: {documented}/{tf} functions = "
      f"{report['docstring_coverage_pct']:.2f}%")

# [6] Function complexity
BRANCH = (ast.If, ast.For, ast.While, ast.Try, ast.With, ast.BoolOp)
comp = []
for s in sources:
    try:
        tree = ast.parse(s)
    except SyntaxError:
        continue
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            comp.append(1 + sum(1 for x in ast.walk(node) if isinstance(x, BRANCH)))
report["complexity_mean"]   = statistics.mean(comp) if comp else 0
report["complexity_median"] = statistics.median(comp) if comp else 0
print(f"\n[6] Function complexity (branches + 1):")
print(f"    mean={report['complexity_mean']:.2f}  median={report['complexity_median']:.0f}  "
      f"n_functions={len(comp)}")

# [7] Largest near-duplicate clusters
top = [c for _, c in near.most_common(5) if c > 1]
report["largest_dup_clusters"] = top
print(f"\n[7] Largest near-duplicate clusters (doc counts): {top if top else 'none'}")

print(f"\n{'='*60}")
print("END REPORT")
print(f"{'='*60}")

with open(OUT_JSON, "w") as f:
    json.dump(report, f, indent=2)
print(f"\nSaved: {OUT_JSON}")


QUALITY REPORT: Ananda100/python-clean-codeparrot  (n=5000 documents)

[1] Syntactic validity:  4278/5000 = 85.56%

[2] Natural-language content:
    Comments:   7.78%
    Docstrings: 8.22%
    NL total:   16.00%

[3] Uniqueness:
    Unique (exact):           5000/5000 = 100.00%
    Unique (near, code-only): 4995/5000 = 99.90%
    Exact duplicate rate: 0.00%
    Near duplicate rate:  0.10%

[4] Document length (characters):
    mean=5170  median=3569  p95=14977  min=19  max=19958

[5] Docstring coverage: 8862/33550 functions = 26.41%

[6] Function complexity (branches + 1):
    mean=2.31  median=1  n_functions=33550

[7] Largest near-duplicate clusters (doc counts): [2, 2, 2, 2, 2]

END REPORT

Saved: corpus_quality_report.json
